# Jeffcott Rotor Simulator
**Author:** Akshat Verma  


----

This notebook implements the **Jeffcott rotor** model of a flexible rotating shaft and simulates its vibration response under four fault conditions:

| Configuration | Physical Meaning |
|---|---|
| `healthy` | Baseline; small residual eccentricity only |
| `unbalance` | Mass unbalance; elevated eccentricity, adjustable phase |
| `crack` | Breathing transverse crack; Mayes-Davies stiffness model |
| `bearing` | Bearing clearance nonlinearity; dead-band contact stiffness |

For each configuration the notebook produces:
- Time-domain displacement traces $x(t)$, $y(t)$
- Shaft orbit plot ($x$ vs $y$)
- FFT spectrum (showing 1×, 2×, 3× harmonics)
- STFT spectrogram and CWT scalogram
- Statistical fault features: RMS, kurtosis, crest factor
- Campbell diagram sweeping from 0 to 3× critical speed

Note: All parameters defined in the `CONFIG` dictionary in Section 1

---

## Section 0 — Imports

All external dependencies are declared here. If any import fails, install with `pip install <package>`.

| Library | Role in this notebook |
|---|---|
| `numpy` | Numerical arrays, FFT, linear algebra |
| `scipy.integrate` | ODE integration (`solve_ivp`, RK45) |
| `scipy.signal` | STFT, Hann window |
| `scipy.stats` | Kurtosis calculation |
| `pywt` | Continuous Wavelet Transform (CWT) |
| `matplotlib` | All plotting |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from scipy.integrate import solve_ivp
from scipy.signal import stft
from scipy.signal.windows import hann
from scipy.stats import kurtosis
import pywt
import warnings
warnings.filterwarnings('ignore')

# Pulot defaults
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.35,
    'lines.linewidth': 1.6,
})

print('All imports successful.')
print(f'NumPy {np.__version__} | PyWavelets {pywt.__version__}')

---
## Section 1 — System Configuration

**Every physical parameter has been defined here.**

### Physical Basis for Default Values

These values are chosen to represent a **small industrial centrifugal pump** impeller shaft:

- **Mass $m = 10$ kg**: A typical 3-stage pump impeller + shaft section between bearings.
- **Stiffness $k = 1\times10^6$ N/m**: From beam theory for a 40 mm steel shaft, 500 mm between bearings: $k = 48EI/L^3$. With $E = 210$ GPa, $I = \pi d^4/64$, this gives $k \approx 1$ MN/m.
- **Damping ratio $\zeta = 0.05$**: Typical for rolling-element bearings. Journal bearings would be 0.1–0.15.
- **Eccentricity $e_0 = 50\,\mu$m**: Residual eccentricity after balancing to ISO G2.5 grade.
- **Operating speed $\Omega = 1500$ rpm**: Synchronous speed for a 4-pole motor at 50 Hz.
- **Critical speed $\omega_n = \sqrt{k/m} \approx 3000$ rpm**: This machine runs at half its critical speed — typical for well-designed pump rotors.

### Fault Parameters

- **Unbalance eccentricity $e_u = 200\,\mu$m**: 4× the healthy value — represents a blade loss or erosion event.
- **Crack ratio $\mu = 0.3$**: 30% crack depth / shaft radius. Measurable 2× component but not yet catastrophic.
- **Bearing clearance $\delta = 100\,\mu$m**: Typical radial clearance for a plain bearing at this shaft size.

### Simulation Parameters

- **Simulation time**: 20 shaft revolutions — long enough for transients to die out and for frequency resolution in FFT.
- **Samples per revolution**: 256 — gives Nyquist frequency of 128× running speed, comfortably above the 3× harmonic of interest.

In [ ]:
# ============================================================
#  SYSTEM CONFIGURATION — Edit parameters here only
# ============================================================
CONFIG = {
    # --- Rotor physical parameters ---
    'm':    10.0,           # Disk mass [kg]
    'k':    1.0e6,          # Shaft stiffness [N/m]
    'zeta': 0.05,           # Damping ratio [-] (c = 2*zeta*sqrt(k*m))

    # --- Baseline eccentricity (healthy rotor) ---
    'e0':   50e-6,          # Residual eccentricity [m] (50 µm)

    # --- Operating speed ---
    'rpm':  1500.0,         # Shaft speed [rpm]

    # --- Fault: Mass Unbalance ---
    'e_unbalance': 200e-6,  # Unbalance eccentricity [m]
    'phi_unbalance': 0.0,   # Unbalance phase angle [rad]

    # --- Fault: Breathing Crack (Mayes-Davies model) ---
    'mu_crack': 0.30,       # Crack depth ratio [-] (0=no crack, 1=full)

    # --- Fault: Bearing Clearance Nonlinearity ---
    'delta_bearing': 100e-6,  # Radial clearance [m] (100 µm)
    'k_contact':    1.0e8,    # Contact stiffness beyond clearance [N/m]

    # --- Simulation parameters ---
    'n_revs':    20,        # Number of shaft revolutions to simulate
    'spr':       256,       # Samples per revolution (controls time resolution)
    'n_transient': 5,       # Revolutions to discard as transient (initial condition decay)

    # --- Campbell diagram ---
    'rpm_campbell_min': 100.0,   # Start speed for Campbell sweep [rpm]
    'rpm_campbell_max': None,    # End speed (None = 3x critical speed, set automatically)
    'n_campbell_points': 80,     # Number of speed points in sweep
    'n_revs_campbell':   10,     # Revolutions per speed point (shorter for speed)
}

# ============================================================
#  Derived quantities
# ============================================================
CONFIG['omega_n'] = np.sqrt(CONFIG['k'] / CONFIG['m'])          # Natural frequency [rad/s]
CONFIG['f_n']     = CONFIG['omega_n'] / (2 * np.pi)              # Natural frequency [Hz]
CONFIG['rpm_n']   = CONFIG['f_n'] * 60.0                         # Critical speed [rpm]
CONFIG['c']       = 2 * CONFIG['zeta'] * np.sqrt(CONFIG['k'] * CONFIG['m'])  # Damping coeff [N·s/m]
CONFIG['Omega']   = CONFIG['rpm'] * (2 * np.pi / 60.0)           # Operating speed [rad/s]
CONFIG['T_rev']   = 2 * np.pi / CONFIG['Omega']                  # Period of one revolution [s]
CONFIG['r']       = CONFIG['Omega'] / CONFIG['omega_n']           # Speed ratio [-]

# Set Campbell max speed if not specified
if CONFIG['rpm_campbell_max'] is None:
    CONFIG['rpm_campbell_max'] = 3.0 * CONFIG['rpm_n']

# Print a summary
print('=' * 55)
print('  JEFFCOTT ROTOR — SYSTEM PARAMETERS')
print('=' * 55)
print(f"  Mass             m  = {CONFIG['m']:.1f} kg")
print(f"  Stiffness        k  = {CONFIG['k']:.3e} N/m")
print(f"  Damping ratio    ζ  = {CONFIG['zeta']:.3f}")
print(f"  Damping coeff    c  = {CONFIG['c']:.1f} N·s/m")
print(f"  Natural freq     ωₙ = {CONFIG['omega_n']:.2f} rad/s")
print(f"  Critical speed      = {CONFIG['rpm_n']:.1f} rpm")
print(f"  Operating speed  Ω  = {CONFIG['rpm']:.1f} rpm")
print(f"  Speed ratio      r  = Ω/ωₙ = {CONFIG['r']:.3f}")
print(f"  Residual eccent. e₀ = {CONFIG['e0']*1e6:.1f} µm")
print('=' * 55)

---
## Section 2 — Analytical Steady-State Solution

### Derivation

For the healthy rotor (constant $k$, linear), assume a particular solution:

$$x_p(t) = X\cos(\Omega t - \psi), \quad y_p(t) = X\sin(\Omega t - \psi)$$

Substituting into the EOM and matching coefficients yields:

$$\boxed{X(r) = \frac{e_0 \cdot r^2}{\sqrt{(1 - r^2)^2 + (2\zeta r)^2}}}$$

$$\boxed{\psi(r) = \arctan\!\left(\frac{2\zeta r}{1 - r^2}\right)}$$

**Phase behaviour is physically important:**
- $r < 1$: $\psi < 90°$ — mass centre $G$ is on the **outside** of the orbit. The rotor bows toward the heavy spot.
- $r = 1$: $\psi = 90°$ — classic resonance phase signature. Amplitude peaks, phase passes through 90°.
- $r > 1$: $\psi \to 180°$ — mass centre $G$ is on the **inside** of the orbit (self-centring).

In [ ]:
def amplitude_response(r_arr, zeta, e):
    """
    Analytical steady-state amplitude of a Jeffcott rotor under synchronous excitation.

    Parameters
    ----------
    r_arr : array_like
        Speed ratio Omega/omega_n [-]
    zeta : float
        Viscous damping ratio [-]
    e : float
        Eccentricity [m]

    Returns
    -------
    X : ndarray
        Vibration amplitude [m]
    psi : ndarray
        Phase lag [rad]
    """
    r = np.asarray(r_arr)
    denom = np.sqrt((1 - r**2)**2 + (2 * zeta * r)**2)
    X   = e * r**2 / denom
    psi = np.arctan2(2 * zeta * r, 1 - r**2)
    return X, psi


# Compute over a dense speed ratio range for the response plot
r_range = np.linspace(0.01, 3.0, 2000)
X_analytical, psi_analytical = amplitude_response(r_range, CONFIG['zeta'], CONFIG['e0'])

# Compute at the operating speed ratio for reference
X_op, psi_op = amplitude_response(CONFIG['r'], CONFIG['zeta'], CONFIG['e0'])
print(f"Analytical steady-state amplitude at {CONFIG['rpm']:.0f} rpm: {X_op*1e6:.2f} µm")
print(f"Phase lag at operating speed:  {np.degrees(psi_op):.1f}°")
print(f"Amplitude at critical speed:   {CONFIG['e0']/(2*CONFIG['zeta'])*1e6:.2f} µm  (= e/(2ζ))")

# --- Plot the analytical response ---
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax1, ax2 = axes

ax1.plot(r_range, X_analytical * 1e6, color='steelblue', label='Amplitude $X(r)$')
ax1.axvline(1.0, color='crimson', ls='--', lw=1.2, label='Critical speed ($r=1$)')
ax1.axvline(CONFIG['r'], color='darkorange', ls=':', lw=1.5,
            label=f'Operating speed ($r={CONFIG["r"]:.2f}$)')
ax1.scatter([CONFIG['r']], [X_op * 1e6], color='darkorange', zorder=5, s=60)
ax1.set_ylabel('Amplitude $X$ [µm]')
ax1.set_title('Jeffcott Rotor — Analytical Frequency Response')
ax1.legend()
ax1.annotate(f'{X_op*1e6:.1f} µm', xy=(CONFIG['r'], X_op * 1e6),
             xytext=(CONFIG['r'] + 0.15, X_op * 1e6 + 5),
             fontsize=9, color='darkorange',
             arrowprops=dict(arrowstyle='->', color='darkorange', lw=1.0))

ax2.plot(r_range, np.degrees(psi_analytical), color='seagreen', label='Phase lag $\\psi(r)$')
ax2.axhline(90, color='crimson', ls='--', lw=1.2, label='90° at resonance')
ax2.axvline(1.0, color='crimson', ls='--', lw=1.2)
ax2.axvline(CONFIG['r'], color='darkorange', ls=':', lw=1.5)
ax2.set_xlabel('Speed ratio $r = \\Omega / \\omega_n$ [-]')
ax2.set_ylabel('Phase lag $\\psi$ [°]')
ax2.set_yticks([0, 45, 90, 135, 180])
ax2.legend()

plt.tight_layout()
plt.savefig('fig_analytical_response.png', bbox_inches='tight')
plt.show()
print()
print('Physical interpretation:')
print(f'  The rotor operates at r = {CONFIG["r"]:.2f} — well below critical speed.')
print(f'  Amplitude is {X_op*1e6:.1f} µm, phase lag is {np.degrees(psi_op):.0f}°.')
print('  The heavy spot and high spot are nearly coincident (sub-critical regime).')
print(f'  At critical speed, amplitude would be {CONFIG["e0"]/(2*CONFIG["zeta"])*1e6:.0f} µm = e₀/(2ζ).')

---
## Section 3 — State-Space Formulation and ODE Functions

`scipy.integrate.solve_ivp` requires a **first-order** ODE system $\dot{\mathbf{q}} = f(t, \mathbf{q})$. Our 2-DOF equations of motion are second-order, so we reduce their order by introducing velocity as an additional state variable.

Define the state vector:

$$\mathbf{q} = \begin{pmatrix} q_1 \\ q_2 \\ q_3 \\ q_4 \end{pmatrix} = \begin{pmatrix} x \\ \dot{x} \\ y \\ \dot{y} \end{pmatrix}$$

Then the system $m\ddot{x} + c\dot{x} + kx = F_x(t)$ becomes:

$$\dot{q}_1 = q_2$$

$$\dot{q}_2 = \frac{1}{m}\left[F_x(t) - c\,q_2 - k_{eff}(t, q_1)\cdot q_1\right]$$

and analogously for $y$. The `solve_ivp` solver evaluates this function at each adaptive time step.

### One Function Per Fault Configuration

Each fault modifies either the **stiffness term** (crack, bearing) or the **forcing term** (unbalance). A unified factory function returns the correct ODE based on the `fault` parameter string. This keeps the integration loop identical across all cases.

### Solver: RK45

RK45 is an **explicit Runge-Kutta method** with adaptive step-size control. It is deployed because:
- The ODE is non-stiff (stiffness ratio is moderate, no huge eigenvalue separation)
- Adaptive stepping automatically refines around the nonlinear contact event (bearing fault)

In [ ]:
def make_ode(cfg, fault='healthy'):
    """
    Factory function: returns an ODE function for the Jeffcott rotor
    under the specified fault configuration.

    Parameters
    ----------
    cfg : dict
        CONFIG dictionary containing all physical parameters.
    fault : str
        One of 'healthy', 'unbalance', 'crack', 'bearing'.

    Returns
    -------
    ode_func : callable
        Function with signature f(t, q) compatible with solve_ivp.
        State vector q = [x, xdot, y, ydot].
    """
    # Unpack parameters into local variables for speed inside the loop
    m       = cfg['m']
    k0      = cfg['k']
    c       = cfg['c']
    Omega   = cfg['Omega']

    # Fault-specific parameters
    if fault == 'healthy':
        e, phi = cfg['e0'], 0.0

    elif fault == 'unbalance':
        e, phi = cfg['e_unbalance'], cfg['phi_unbalance']

    elif fault == 'crack':
        e, phi = cfg['e0'], 0.0
        mu = cfg['mu_crack']

    elif fault == 'bearing':
        e, phi  = cfg['e0'], 0.0
        delta   = cfg['delta_bearing']
        k_c     = cfg['k_contact']

    else:
        raise ValueError(f"Unknown fault type: '{fault}'. "
                         "Choose from: healthy, unbalance, crack, bearing")

    # ---------------------------------------------------------------
    #  Healthy / Unbalance: linear, constant coefficients
    # ---------------------------------------------------------------
    if fault in ('healthy', 'unbalance'):
        def ode_func(t, q):
            x, xd, y, yd = q
            theta = Omega * t + phi
            # Centrifugal forcing amplitude
            F0   = m * e * Omega**2
            Fx   = F0 * np.cos(theta)
            Fy   = F0 * np.sin(theta)
            xdd  = (Fx - c * xd - k0 * x) / m
            ydd  = (Fy - c * yd - k0 * y) / m
            return [xd, xdd, yd, ydd]

    # ---------------------------------------------------------------
    #  Breathing Crack: Mayes-Davies parametric stiffness
    #  k(t) = k0 * (1 - mu * cos(Omega*t))
    #  Stiffness reduces most when crack is fully open (theta=0, top)
    # ---------------------------------------------------------------
    elif fault == 'crack':
        def ode_func(t, q):
            x, xd, y, yd = q
            theta  = Omega * t
            k_eff  = k0 * (1.0 - mu * np.cos(theta))   # Mayes-Davies
            F0     = m * e * Omega**2
            Fx     = F0 * np.cos(theta)
            Fy     = F0 * np.sin(theta)
            xdd    = (Fx - c * xd - k_eff * x) / m
            ydd    = (Fy - c * yd - k_eff * y) / m
            return [xd, xdd, yd, ydd]

    # ---------------------------------------------------------------
    #  Bearing Clearance: piecewise-linear restoring force
    #  F_bearing = -k0*x - k_c*(|x|-delta)*sgn(x)  if |x| > delta
    #            = -k0*x                              if |x| <= delta
    # ---------------------------------------------------------------
    elif fault == 'bearing':
        def ode_func(t, q):
            x, xd, y, yd = q
            theta = Omega * t
            F0    = m * e * Omega**2
            Fx    = F0 * np.cos(theta)
            Fy    = F0 * np.sin(theta)

            # Piecewise restoring force in x
            overlap_x = np.abs(x) - delta
            if overlap_x > 0:
                Rx = -k0 * x - k_c * overlap_x * np.sign(x)
            else:
                Rx = -k0 * x

            # Piecewise restoring force in y
            overlap_y = np.abs(y) - delta
            if overlap_y > 0:
                Ry = -k0 * y - k_c * overlap_y * np.sign(y)
            else:
                Ry = -k0 * y

            xdd = (Fx + Rx - c * xd) / m
            ydd = (Fy + Ry - c * yd) / m
            return [xd, xdd, yd, ydd]

    return ode_func


def run_simulation(cfg, fault='healthy'):
    """
    Integrate the Jeffcott rotor ODE for the specified fault configuration.

    Parameters
    ----------
    cfg : dict
        Configuration dictionary.
    fault : str
        Fault configuration identifier.

    Returns
    -------
    t : ndarray
        Time array (transient-free) [s]
    x : ndarray
        Lateral displacement in X [m]
    y : ndarray
        Lateral displacement in Y [m]
    """
    T_rev   = cfg['T_rev']
    n_revs  = cfg['n_revs']
    spr     = cfg['spr']
    n_trans = cfg['n_transient']

    # Build time array: uniform points over all revolutions
    dt       = T_rev / spr
    t_end    = n_revs * T_rev
    t_eval   = np.arange(0, t_end, dt)

    # Zero initial conditions (rotor starts at rest at centreline)
    q0 = [0.0, 0.0, 0.0, 0.0]

    ode = make_ode(cfg, fault)
    sol = solve_ivp(
        ode,
        t_span=(0, t_end),
        y0=q0,
        method='RK45',
        t_eval=t_eval,
        rtol=1e-8,
        atol=1e-10,
    )

    # Discard transient revolutions
    i_start = n_trans * spr
    t = sol.t[i_start:]
    x = sol.y[0, i_start:]
    y = sol.y[2, i_start:]

    return t, x, y


print('ODE functions and simulation runner defined.')
print('Running healthy rotor simulation to verify...')
t_h, x_h, y_h = run_simulation(CONFIG, fault='healthy')

# Quick validation against analytical solution
X_num   = np.max(np.abs(x_h))
X_anal, _ = amplitude_response(CONFIG['r'], CONFIG['zeta'], CONFIG['e0'])
error_pct = abs(X_num - X_anal) / X_anal * 100
print(f'Numerical amplitude:   {X_num*1e6:.3f} µm')
print(f'Analytical amplitude:  {X_anal*1e6:.3f} µm')
print(f'Error:                 {error_pct:.4f}%  (< 0.1% expected for RK45)')

---
## Section 4 — Time Domain, Orbit, and FFT Analysis Functions

### Time Domain Signals

$x(t)$ and $y(t)$ are the **lateral shaft displacements** in the two transverse directions. In a real machine, these are what your **eddy-current proximity probes** measure — non-contacting sensors mounted at 90° to each other at the bearing plane.

### Orbit Plot

The orbit is the trajectory of the shaft centreline $C$ in the $(x,y)$ plane. Under steady synchronous excitation (healthy rotor), the orbit is a **perfect circle** (forward whirl, $x$ and $y$ have equal amplitude and 90° phase difference). Deviations from circular orbits are diagnostic:
- **Elliptical orbit**: Anisotropic bearing stiffness or slight misalignment
- **Figure-of-eight / looped orbit**: Sub-harmonic response, rub, or bearing nonlinearity
- **Inner loop in orbit**: Presence of 2× component (crack signature)

### FFT Spectrum

The Fast Fourier Transform reveals the **frequency content** of the vibration signal. We use a Hann window to reduce spectral leakage — without it, energy from the main peak bleeds into adjacent bins and masks low-amplitude harmonics.

**Key frequencies to watch:**
- **1× (1 × running speed)**: Synchronous component. Present in all cases. Dominated by unbalance.
- **2×**: Indicates asymmetric stiffness, misalignment, or (diagnostically) a breathing crack.
- **3× and higher**: Rub, bearing looseness, or strong nonlinearity.
- **½×**: Oil whirl precursor to oil whip instability (not in this model, but important in your turbines).

In [ ]:
def compute_fft(t, x, cfg):
    """
    Compute windowed FFT of a displacement signal.
    Returns frequency in units of running speed (order spectrum).

    Parameters
    ----------
    t : ndarray
        Time array [s]
    x : ndarray
        Displacement signal [m]
    cfg : dict
        CONFIG dictionary

    Returns
    -------
    orders : ndarray
        Frequency in multiples of running speed (1× = 1.0)
    amplitude : ndarray
        Single-sided amplitude spectrum [m]
    """
    N      = len(x)
    dt     = t[1] - t[0]
    fs     = 1.0 / dt                            # Sampling frequency [Hz]
    f_run  = cfg['Omega'] / (2 * np.pi)          # Running frequency [Hz]

    window = np.hanning(N)                        # Hann window to reduce leakage
    X_fft  = np.fft.rfft(x * window)             # Real FFT (one-sided)
    freqs  = np.fft.rfftfreq(N, d=dt)            # Frequency array [Hz]

    # Convert to amplitude spectrum (correct for window and one-sided)
    amplitude = 2.0 * np.abs(X_fft) / N
    amplitude[0]  /= 2.0   # DC component not doubled
    amplitude[-1] /= 2.0   # Nyquist component not doubled

    # Normalise frequency axis to running speed orders
    orders = freqs / f_run

    return orders, amplitude


def compute_statistical_features(x):
    """
    Compute scalar fault-sensitive statistical features from a vibration signal.

    Parameters
    ----------
    x : ndarray
        Vibration signal [m] (or any consistent unit)

    Returns
    -------
    features : dict
        Dictionary of feature name → value

    Notes
    -----
    RMS         : Energy content — increases with unbalance severity.
    Kurtosis    : Impulsiveness — peaks sharply for bearing defects (kurtosis > 4
                  is a common alarm threshold in ISO 13373).
    Crest factor: Peak / RMS — amplifies impulsive events.
    Peak-to-peak: Overall dynamic range.
    """
    rms         = np.sqrt(np.mean(x**2))
    kurt        = kurtosis(x, fisher=False)       # Fisher=False gives 3.0 for Gaussian
    peak        = np.max(np.abs(x))
    crest       = peak / rms if rms > 0 else 0.0
    p2p         = np.ptp(x)

    return {
        'RMS [µm]':          rms * 1e6,
        'Kurtosis [-]':      kurt,
        'Crest Factor [-]':  crest,
        'Peak-to-peak [µm]': p2p * 1e6,
    }


def plot_rotor_response(t, x, y, cfg, fault_label='Healthy', color='steelblue'):
    """
    Produce a 4-panel diagnostic plot for a single fault configuration:
    (1) Time domain x(t),  (2) Time domain y(t),
    (3) Orbit x vs y,      (4) FFT order spectrum.

    Parameters
    ----------
    t, x, y     : ndarrays from run_simulation()
    cfg         : CONFIG dictionary
    fault_label : str for plot title
    color       : line colour
    """
    f_run   = cfg['Omega'] / (2 * np.pi)
    T_rev   = cfg['T_rev']
    orders, amp = compute_fft(t, x, cfg)

    fig = plt.figure(figsize=(14, 9))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

    ax_x    = fig.add_subplot(gs[0, :2])  # x(t) — wide
    ax_y    = fig.add_subplot(gs[1, :2])  # y(t) — wide
    ax_orb  = fig.add_subplot(gs[0, 2])   # orbit
    ax_fft  = fig.add_subplot(gs[1, 2])   # FFT

    # --- Time domain: show first 5 revolutions for clarity ---
    mask = (t - t[0]) <= 5 * T_rev
    ax_x.plot((t[mask] - t[0]), x[mask] * 1e6, color=color)
    ax_x.set_xlabel('Time [s]')
    ax_x.set_ylabel('Displacement $x$ [µm]')
    ax_x.set_title(f'{fault_label} — $x(t)$')

    ax_y.plot((t[mask] - t[0]), y[mask] * 1e6, color=color)
    ax_y.set_xlabel('Time [s]')
    ax_y.set_ylabel('Displacement $y$ [µm]')
    ax_y.set_title(f'{fault_label} — $y(t)$')

    # --- Orbit: last 3 revolutions (steady state) ---
    n_pts = 3 * cfg['spr']
    ax_orb.plot(x[-n_pts:] * 1e6, y[-n_pts:] * 1e6, color=color, lw=1.2)
    ax_orb.set_aspect('equal')
    ax_orb.set_xlabel('$x$ [µm]')
    ax_orb.set_ylabel('$y$ [µm]')
    ax_orb.set_title('Shaft Orbit')
    ax_orb.axhline(0, color='k', lw=0.5)
    ax_orb.axvline(0, color='k', lw=0.5)

    # --- FFT: plot to 5× order, mark 1×, 2×, 3× ---
    mask_f = orders <= 5.5
    ax_fft.semilogy(orders[mask_f], amp[mask_f] * 1e6 + 1e-6,
                    color=color, lw=1.2)
    for nx, label in zip([1, 2, 3], ['1×', '2×', '3×']):
        ax_fft.axvline(nx, color='crimson', ls='--', lw=0.9, alpha=0.7)
        ax_fft.text(nx + 0.05, ax_fft.get_ylim()[1] if ax_fft.get_ylim()[1] > 1 else 10,
                    label, color='crimson', fontsize=8)
    ax_fft.set_xlabel('Order (× running speed)')
    ax_fft.set_ylabel('Amplitude [µm]')
    ax_fft.set_title('FFT Order Spectrum')
    ax_fft.set_xlim(0, 5.5)

    fig.suptitle(f'Jeffcott Rotor — {fault_label} Configuration | '
                 f'{cfg["rpm"]:.0f} rpm | ζ = {cfg["zeta"]:.2f}',
                 fontsize=13, fontweight='bold')
    plt.savefig(f'fig_{fault_label.lower().replace(" ","_")}_response.png',
                bbox_inches='tight')
    plt.show()


print('Analysis and plotting functions defined.')

---
## Section 5 — Case 1: Healthy Rotor

The healthy rotor has only a small residual eccentricity $e_0 = 50\,\mu$m. Under synchronous excitation, the steady-state response is purely at 1× running speed:

- $x(t)$ and $y(t)$ are perfect sinusoids at running frequency, 90° out of phase
- The orbit is a **circle** (or near-circle)
- The FFT shows a **single spike at 1×**; all other orders are at the noise floor

This is **baseline** — any deviation from this signature in a real machine is a fault indicator.

In [ ]:
print('Simulating HEALTHY rotor...')
t_h, x_h, y_h = run_simulation(CONFIG, fault='healthy')
plot_rotor_response(t_h, x_h, y_h, CONFIG, fault_label='Healthy Rotor', color='steelblue')

feat_h = compute_statistical_features(x_h)
print('\nStatistical features — Healthy rotor:')
for k, v in feat_h.items():
    print(f'  {k:<25s}: {v:.4f}')

print('\nPhysical interpretation:')
print('  Single 1× spike confirms synchronous, unidirectional excitation.')
print(f'  Circular orbit with radius ≈ {np.max(np.abs(x_h))*1e6:.1f} µm confirms')
print('  isotropic stiffness (equal x and y amplitudes).')
print('  Kurtosis ≈ 1.5 (sinusoidal signal) — no impulsive events.')

---
## Section 6 — Case 2: Mass Unbalance Fault

Mass unbalance occurs when the centre of mass $G$ is not coincident with the geometric centre $C$ of the rotor.
The unbalance force $F = me\Omega^2$ scales with $\Omega^2$, so this fault becomes more damaging as speed increases.

### Spectral Signature

**1× dominance** — the only difference from the healthy case is the amplitude. **Elevated 1× with a smooth, round orbit** signals unbalance. If the 1× is elevated but the orbit is irregular, misalignment or looseness is suspected along with the unbalance.

In [ ]:
print('Simulating MASS UNBALANCE fault...')
t_u, x_u, y_u = run_simulation(CONFIG, fault='unbalance')
plot_rotor_response(t_u, x_u, y_u, CONFIG, fault_label='Mass Unbalance', color='darkorange')

feat_u = compute_statistical_features(x_u)
print('\nStatistical features — Mass Unbalance:')
for k, v in feat_u.items():
    print(f'  {k:<25s}: {v:.4f}')

print(f'\nEccentricity ratio (fault/healthy): {CONFIG["e_unbalance"]/CONFIG["e0"]:.1f}×')
print(f'Expected amplitude ratio:           {CONFIG["e_unbalance"]/CONFIG["e0"]:.1f}× (linear scaling)')
print('\nPhysical interpretation:')
print('  Larger orbit radius — same circular shape.')
print('  Only the 1× component changes. Spectrum shape identical to healthy.')
print('  In practice: ISO 10816 velocity alarm limits would be breached.')

---
## Section 7 — Case 3: Breathing Crack Fault (Mayes-Davies Model)

### Physical Mechanism

A transverse crack on a rotating shaft is subject to gravity bending. As the shaft rotates:
- **Top of rotation (crack faces pointing up)**: Gravity load opens the crack → stiffness is reduced
- **Bottom of rotation (crack faces pointing down)**: Gravity load closes the crack → stiffness is restored

This **opening and closing once per revolution** (breathing) makes the shaft stiffness periodic at 1×:

$$k(\theta) = k_0\left(1 - \mu\cos\theta\right), \quad \theta = \Omega t$$

### 2× Appears in the Spectrum

The equation of motion with the Mayes-Davies stiffness is:

$$m\ddot{x} + c\dot{x} + k_0(1 - \mu\cos\Omega t)\,x = me\Omega^2\cos\Omega t$$

The stiffness term $k_0\mu\cos(\Omega t)\cdot x$ acts as a **parametric excitation**. When $x$ is already vibrating at 1×, the product $\cos(\Omega t) \cdot \cos(\Omega t)$ generates a component at 2× (from the identity $\cos^2\theta = \frac{1}{2}(1 + \cos 2\theta)$).

**Key diagnostic feature: the 2× / 1× amplitude ratio increases with crack depth $\mu$.** This is the primary crack indicator used in condition monitoring practice.

In [ ]:
print('Simulating BREATHING CRACK fault (Mayes-Davies, µ={:.2f})...'.format(CONFIG['mu_crack']))
t_c, x_c, y_c = run_simulation(CONFIG, fault='crack')
plot_rotor_response(t_c, x_c, y_c, CONFIG, fault_label='Breathing Crack', color='crimson')

feat_c = compute_statistical_features(x_c)
print('\nStatistical features — Breathing Crack:')
for k, v in feat_c.items():
    print(f'  {k:<25s}: {v:.4f}')

# Extract 1× and 2× amplitudes from FFT
orders_c, amp_c = compute_fft(t_c, x_c, CONFIG)

def get_order_amplitude(orders, amp, target_order, bandwidth=0.1):
    """Return peak amplitude within ±bandwidth of a target order."""
    mask = np.abs(orders - target_order) < bandwidth
    if not np.any(mask):
        return 0.0
    return np.max(amp[mask])

amp_1x = get_order_amplitude(orders_c, amp_c, 1.0)
amp_2x = get_order_amplitude(orders_c, amp_c, 2.0)
print(f'\n1× amplitude: {amp_1x*1e6:.3f} µm')
print(f'2× amplitude: {amp_2x*1e6:.4f} µm')
print(f'2×/1× ratio:  {amp_2x/amp_1x:.4f}  (grows with crack depth µ)')
print('\nPhysical interpretation:')
print('  Orbit shows inner loop — hallmark of a 2× component.')
print('  2× spike in FFT is the primary crack indicator.')
print('  Higher µ → larger inner loop → larger 2×/1× ratio.')
print('  Orbit shape: figure-of-8 if 2× amplitude is significant.')

---
## Section 8 — Case 4: Bearing Clearance Nonlinearity

### Physical Mechanism

Every bearing has a **radial clearance** $\delta$ — the gap between the journal (rotating inner part) and the bearing shell (fixed outer part). Under normal operation:
- The shaft centre orbits within this clearance — the restoring force is linear (oil film or rolling elements)
- As bearing wear progresses, $\delta$ increases

When vibration amplitude exceeds the clearance ($|x| > \delta$), the journal physically **impacts the bearing shell**, creating:
1. A sudden increase in restoring force (contact stiffness $k_c \gg k_0$)
2. An impulsive force at the moment of contact

### Sub- and Super-Harmonics Appear

The piecewise-linear stiffness is equivalent to periodically switching between two linear systems. This **nonlinear switching** generates:
- **Super-harmonics** (2×, 3×, ...): Integer multiples of the forcing frequency, generated by the nonlinearity through the Volterra series expansion of the piecewise function
- **Sub-harmonics** (½×, ¼×, ...): Only at certain speed/clearance ratios where period-doubling bifurcations occur

In [ ]:
print('Simulating BEARING CLEARANCE nonlinearity (δ={:.0f} µm)...'.format(
    CONFIG['delta_bearing'] * 1e6))
t_b, x_b, y_b = run_simulation(CONFIG, fault='bearing')
plot_rotor_response(t_b, x_b, y_b, CONFIG, fault_label='Bearing Clearance', color='seagreen')

feat_b = compute_statistical_features(x_b)
print('\nStatistical features — Bearing Clearance:')
for k, v in feat_b.items():
    print(f'  {k:<25s}: {v:.4f}')

orders_b, amp_b = compute_fft(t_b, x_b, CONFIG)
amp_1x_b = get_order_amplitude(orders_b, amp_b, 1.0)
amp_2x_b = get_order_amplitude(orders_b, amp_b, 2.0)
amp_3x_b = get_order_amplitude(orders_b, amp_b, 3.0)
print(f'\n1× amplitude: {amp_1x_b*1e6:.3f} µm')
print(f'2× amplitude: {amp_2x_b*1e6:.4f} µm')
print(f'3× amplitude: {amp_3x_b*1e6:.4f} µm')
print('\nPhysical interpretation:')
print('  Orbit is clipped/flattened when shaft contacts bearing shell.')
print('  Multiple harmonics in FFT — nonlinear signature.')
print('  Kurtosis elevated above healthy case: impulsive contacts.')
print(f'  Healthy kurtosis: {feat_h["Kurtosis [-]"]:.2f} | Bearing fault: {feat_b["Kurtosis [-]"]:.2f}')

---
## Section 9 — Time-Frequency Analysis: STFT and CWT

### STFT (Short-Time Fourier Transform)

Divides the signal into overlapping short windows and computes the FFT of each. The time-frequency resolution is a fixed trade-off set by the window length: **long window → good frequency resolution, poor time resolution**

We use `scipy.signal.stft` with a Hann window.

### CWT (Continuous Wavelet Transform)

The CWT uses **variable-length basis functions** (wavelets) that automatically adapt their time-frequency resolution: at high frequencies it uses short, narrow wavelets (good time resolution); at low frequencies it uses long, wide wavelets (good frequency resolution). This is better than STFT for multi-scale rotating machinery signals.

We use `pywt.cwt` with the **Morlet wavelet** (`cmor` — complex Morlet) — the standard choice for rotating machinery because its shape resembles a short burst of oscillation, matching the impulsive physics of bearing contact and crack breathing.

In [ ]:
def plot_time_frequency(t, x, cfg, fault_label='Healthy', color_map='viridis'):
    """
    Compute and plot STFT spectrogram and CWT scalogram for a vibration signal.

    Parameters
    ----------
    t, x      : time and signal arrays from run_simulation()
    cfg       : CONFIG dictionary
    fault_label: string for plot title
    color_map : matplotlib colourmap name
    """
    dt     = t[1] - t[0]
    fs     = 1.0 / dt
    f_run  = cfg['Omega'] / (2 * np.pi)   # Running frequency [Hz]

    # ---------------------------------------------------------------
    #  STFT
    # ---------------------------------------------------------------
    nperseg  = cfg['spr'] * 2     # Window = 2 revolutions
    noverlap = int(nperseg * 0.75) # 75% overlap
    f_stft, t_stft, Zxx = stft(x, fs=fs, window='hann',
                                nperseg=nperseg, noverlap=noverlap)
    # Convert frequency axis to orders
    orders_stft = f_stft / f_run
    # Convert time to revolutions
    revs_stft   = (t_stft) * f_run

    # ---------------------------------------------------------------
    #  CWT — Morlet wavelet
    # ---------------------------------------------------------------
    # Scale range: cover 0.5× to 5× running speed
    # For Morlet wavelet: pseudo-frequency ≈ center_freq / (scale * dt)
    wavelet     = 'cmor1.5-1.0'           # Complex Morlet: bandwidth=1.5, center=1.0
    center_freq = pywt.central_frequency(wavelet)
    f_min_cwt   = 0.3 * f_run
    f_max_cwt   = 5.0 * f_run
    scale_max   = center_freq / (f_min_cwt * dt)
    scale_min   = center_freq / (f_max_cwt * dt)
    scales      = np.geomspace(scale_min, scale_max, 256)  # Log-spaced scales

    coeffs, freqs_cwt = pywt.cwt(x, scales, wavelet, sampling_period=dt)
    orders_cwt = freqs_cwt / f_run
    revs_cwt   = (t - t[0]) * f_run

    # ---------------------------------------------------------------
    #  Plot
    # ---------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # STFT
    mask_o = (orders_stft >= 0) & (orders_stft <= 5)
    pcm1 = axes[0].pcolormesh(
        revs_stft, orders_stft[mask_o],
        np.abs(Zxx[mask_o, :]) * 1e6,
        shading='gouraud', cmap=color_map
    )
    for nx in [1, 2, 3]:
        axes[0].axhline(nx, color='white', ls='--', lw=0.8, alpha=0.6)
        axes[0].text(revs_stft[-1] * 0.98, nx + 0.08, f'{nx}×',
                     color='white', fontsize=8, ha='right')
    axes[0].set_xlabel('Revolutions')
    axes[0].set_ylabel('Order (× running speed)')
    axes[0].set_title(f'STFT Spectrogram — {fault_label}')
    plt.colorbar(pcm1, ax=axes[0], label='Amplitude [µm]')

    # CWT
    mask_cwt = (orders_cwt >= 0.3) & (orders_cwt <= 5)
    power_cwt = np.abs(coeffs[mask_cwt, :]) * 1e6
    pcm2 = axes[1].pcolormesh(
        revs_cwt, orders_cwt[mask_cwt],
        power_cwt,
        shading='gouraud', cmap=color_map
    )
    for nx in [1, 2, 3]:
        axes[1].axhline(nx, color='white', ls='--', lw=0.8, alpha=0.6)
        axes[1].text(revs_cwt[-1] * 0.98, nx + 0.08, f'{nx}×',
                     color='white', fontsize=8, ha='right')
    axes[1].set_xlabel('Revolutions')
    axes[1].set_ylabel('Order (× running speed)')
    axes[1].set_title(f'CWT Scalogram (Morlet) — {fault_label}')
    plt.colorbar(pcm2, ax=axes[1], label='Amplitude [µm]')

    plt.suptitle(f'Time-Frequency Analysis — {fault_label}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'fig_tf_{fault_label.lower().replace(" ","_")}.png', bbox_inches='tight')
    plt.show()


# Run for each fault
for fault, t_arr, x_arr, label, cmap in [
        ('healthy',  t_h, x_h, 'Healthy Rotor',    'Blues'),
        ('unbalance',t_u, x_u, 'Mass Unbalance',   'Oranges'),
        ('crack',    t_c, x_c, 'Breathing Crack',  'Reds'),
        ('bearing',  t_b, x_b, 'Bearing Clearance','Greens'),
]:
    print(f'\nTime-frequency analysis: {label}')
    plot_time_frequency(t_arr, x_arr, CONFIG, fault_label=label, color_map=cmap)

---
## Section 10 — Statistical Features

Time-domain statistical features are **scalar descriptors** that condense the entire vibration signal into a single number. They are:
- Fast to compute (no FFT needed)
- Well-suited as inputs to machine learning classifiers
- The basis of many ISO condition monitoring standards

### Feature Definitions and Physical Meaning

| Feature | Formula | Fault sensitivity |
|---|---|---|
| **RMS** | $\sqrt{\frac{1}{N}\sum x_i^2}$ | Overall energy; increases with unbalance severity |
| **Kurtosis** | $\frac{E[(x-\mu)^4]}{\sigma^4}$ | Impulsiveness; pure sinusoid = 1.5, Gaussian = 3.0, impulsive > 4 |
| **Crest Factor** | $x_{peak} / x_{RMS}$ | Ratio of peak to energy; high for early bearing defects |
| **Peak-to-peak** | $x_{max} - x_{min}$ | Dynamic range; directly compared to ISO 7919 shaft limits |

**Important:** Kurtosis and crest factor are **normalized** features — they don't change with machine size or speed, which makes them transferable across machines. RMS and peak-to-peak are **dimensional**.

In [ ]:
import pandas as pd

# Compute features for all fault configurations
fault_cases = {
    'Healthy':          compute_statistical_features(x_h),
    'Mass Unbalance':   compute_statistical_features(x_u),
    'Breathing Crack':  compute_statistical_features(x_c),
    'Bearing Clearance':compute_statistical_features(x_b),
}

df_features = pd.DataFrame(fault_cases).T
df_features.index.name = 'Configuration'

print('\n' + '='*70)
print('  STATISTICAL FEATURE COMPARISON TABLE')
print('='*70)
print(df_features.to_string(float_format=lambda x: f'{x:.4f}'))
print('='*70)

# --- Visual bar chart ---
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
colors = ['steelblue', 'darkorange', 'crimson', 'seagreen']
cases  = list(fault_cases.keys())

for i, (feat_name, ax) in enumerate(zip(df_features.columns, axes)):
    vals = df_features[feat_name].values
    bars = ax.bar(range(len(cases)), vals, color=colors, edgecolor='k', linewidth=0.7)
    ax.set_xticks(range(len(cases)))
    ax.set_xticklabels(['Healthy', 'Unbalance', 'Crack', 'Bearing'],
                       rotation=30, ha='right', fontsize=9)
    ax.set_title(feat_name, fontsize=10)
    # Annotate bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Statistical Feature Comparison — All Fault Configurations',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_statistical_features.png', bbox_inches='tight')
plt.show()

print('\nPhysical interpretation:')
print('  RMS:          Unbalance shows highest energy (large eccentricity).')
print('  Kurtosis:     Bearing fault shows highest impulsiveness (contact impacts).')
print('  Crest factor: Bearing fault peaks sharply — classic early-fault indicator.')
print('  Peak-to-peak: Compare to ISO 7919 displacement limits for your shaft size.')

---
## Section 11 — Campbell Diagram

### What is a Campbell Diagram?

A **Campbell diagram** (also called a speed-frequency diagram or interference diagram) plots:
- **X-axis**: Shaft rotational speed (rpm or rad/s)
- **Y-axis**: Vibration amplitude (or sometimes frequency)
- Shows how the response amplitude evolves as the machine runs up from rest to operating speed

For a Jeffcott rotor, the Campbell diagram collapses to a single sweep of the amplitude response function $X(r)$ — but for multi-stage turbines and compressors with multiple natural frequencies and multiple harmonic excitations, the Campbell diagram becomes critical for ensuring that no resonance crossing occurs at operating speed.

### Engineering Significance

**The critical speed is the most important parameter in rotor design.** Rules of thumb:
- For flexible rotors (most industrial turbomachinery): design the first critical speed to be **below** operating speed, and pass through it quickly during startup/shutdown
- For rigid rotors (small high-speed machines): design the first critical speed to be **above** operating speed (run subcritically)

**Why does amplitude peak so sharply at critical speed?** At $r = 1$, the forcing frequency exactly matches the natural frequency. Energy is fed into the system at exactly the rate it naturally wants to oscillate, and the damping is the only limit on amplitude growth. With $\zeta = 0.05$, the peak amplitude is $1/(2\zeta) = 10\times$ the eccentricity.

**What happens above critical speed?** The amplitude comes back down (the rotor self-centres), and the phase shifts by ~180°. This phase reversal through resonance is the key measurement used in **modal balancing** of flexible rotors.

In [ ]:
def run_campbell(cfg, fault='healthy', n_points=None, n_revs=None):
    """
    Sweep shaft speed from rpm_campbell_min to rpm_campbell_max,
    simulate the rotor at each speed, and extract the steady-state
    vibration amplitude.

    Parameters
    ----------
    cfg     : CONFIG dictionary
    fault   : fault configuration string
    n_points: number of speed points (default from cfg)
    n_revs  : revolutions per speed point (default from cfg)

    Returns
    -------
    rpm_array  : ndarray of shaft speeds [rpm]
    amp_array  : ndarray of vibration amplitudes [m]
    """
    n_points = n_points or cfg['n_campbell_points']
    n_revs   = n_revs   or cfg['n_revs_campbell']

    rpm_array = np.linspace(cfg['rpm_campbell_min'],
                            cfg['rpm_campbell_max'], n_points)
    amp_array = np.zeros(n_points)

    # Build a temporary config copy with modified speed for each point
    cfg_sweep = cfg.copy()
    cfg_sweep['n_revs']    = n_revs
    cfg_sweep['n_transient'] = max(2, n_revs // 3)

    for i, rpm in enumerate(rpm_array):
        cfg_sweep['rpm']    = rpm
        cfg_sweep['Omega']  = rpm * (2 * np.pi / 60.0)
        cfg_sweep['T_rev']  = 2 * np.pi / cfg_sweep['Omega']
        cfg_sweep['r']      = cfg_sweep['Omega'] / cfg['omega_n']
        cfg_sweep['c']      = 2 * cfg['zeta'] * np.sqrt(cfg['k'] * cfg['m'])

        try:
            _, x_s, _ = run_simulation(cfg_sweep, fault=fault)
            amp_array[i] = np.sqrt(np.mean(x_s**2))   # RMS amplitude
        except Exception:
            amp_array[i] = np.nan

        if (i + 1) % 20 == 0:
            print(f'  Progress: {i+1}/{n_points} speed points done...')

    return rpm_array, amp_array


print('Running Campbell diagram sweep — this may take 1-2 minutes...')
print(f'Speed range: {CONFIG["rpm_campbell_min"]:.0f} — {CONFIG["rpm_campbell_max"]:.0f} rpm')
print(f'Critical speed: {CONFIG["rpm_n"]:.1f} rpm')

rpm_camp, amp_camp_h = run_campbell(CONFIG, fault='healthy')
_, amp_camp_c         = run_campbell(CONFIG, fault='crack')

print('Campbell sweep complete.')

# --- Also compute analytical response for overlay ---
r_campbell = rpm_camp / CONFIG['rpm_n']
X_anal_camp, _ = amplitude_response(r_campbell, CONFIG['zeta'], CONFIG['e0'])

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(rpm_camp, amp_camp_h * 1e6, color='steelblue', lw=2, label='Healthy (numerical)')
ax.plot(rpm_camp, X_anal_camp * 1e6 / np.sqrt(2), color='steelblue',
        ls='--', lw=1.2, alpha=0.6, label='Healthy (analytical RMS)')
ax.plot(rpm_camp, amp_camp_c * 1e6, color='crimson', lw=2, label='Breathing Crack (µ={:.2f})'.format(CONFIG['mu_crack']))

# Mark critical speed
ax.axvline(CONFIG['rpm_n'], color='k', ls='--', lw=1.5, label=f'Critical speed ({CONFIG["rpm_n"]:.0f} rpm)')
ax.axvline(CONFIG['rpm'],   color='darkorange', ls=':', lw=2.0,
           label=f'Operating speed ({CONFIG["rpm"]:.0f} rpm)')

# Mark half-critical (parametric resonance region for crack)
ax.axvline(CONFIG['rpm_n'] / 2, color='crimson', ls=':', lw=1.0, alpha=0.6,
           label=f'½× critical ({CONFIG["rpm_n"]/2:.0f} rpm) — crack resonance')

# Annotation box
peak_amp_h = np.nanmax(amp_camp_h) * 1e6
peak_rpm_h = rpm_camp[np.nanargmax(amp_camp_h)]
ax.annotate(f'Peak: {peak_amp_h:.1f} µm\n@ {peak_rpm_h:.0f} rpm',
            xy=(peak_rpm_h, peak_amp_h),
            xytext=(peak_rpm_h + 200, peak_amp_h * 0.7),
            fontsize=9,
            arrowprops=dict(arrowstyle='->', color='steelblue'))

ax.set_xlabel('Shaft Speed [rpm]')
ax.set_ylabel('Vibration Amplitude RMS [µm]')
ax.set_title('Campbell Diagram — Jeffcott Rotor\nAmplitude vs Speed Sweep', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim(rpm_camp[0], rpm_camp[-1])

plt.tight_layout()
plt.savefig('fig_campbell_diagram.png', bbox_inches='tight')
plt.show()

print('\nPhysical interpretation:')
print(f'  Critical speed: {CONFIG["rpm_n"]:.1f} rpm — peak amplitude = {peak_amp_h:.1f} µm = e/(2ζ)/√2')
print(f'  Operating speed ({CONFIG["rpm"]:.0f} rpm) is at r = {CONFIG["r"]:.2f} of critical.')
print('  Crack case shows secondary peak near ½× critical — parametric resonance.')
print('  Above critical speed, amplitude falls: rotor self-centres (supercritical regime).')
print('  API 610: operating speed must be outside ±20% of critical speed band.')
print(f'  API exclusion zone: {CONFIG["rpm_n"]*0.8:.0f} – {CONFIG["rpm_n"]*1.2:.0f} rpm.')

---
## Section 12 — Summary: Fault Fingerprints


In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

fault_data = [
    (t_h, x_h, 'Healthy Rotor',    'steelblue'),
    (t_u, x_u, 'Mass Unbalance',   'darkorange'),
    (t_c, x_c, 'Breathing Crack',  'crimson'),
    (t_b, x_b, 'Bearing Clearance','seagreen'),
]

expected_signatures = [
    'Pure 1× — reference baseline',
    'Elevated 1× only — synchronous, linear',
    '1× dominant + visible 2× — parametric excitation',
    '1× + 2× + 3× + higher orders — nonlinear clipping',
]

for ax, (t, x, label, color), sig in zip(axes, fault_data, expected_signatures):
    orders, amp = compute_fft(t, x, CONFIG)
    mask = orders <= 6.0
    ax.bar(orders[mask], amp[mask] * 1e6, width=0.04, color=color,
           alpha=0.85, edgecolor='none')
    for nx in [1, 2, 3, 4, 5]:
        ax.axvline(nx, color='k', ls='--', lw=0.6, alpha=0.4)
    ax.set_ylabel('Amplitude [µm]', fontsize=10)
    ax.set_title(f'{label}  |  {sig}', fontsize=10, fontweight='bold', color=color)
    ax.set_xticks(range(7))
    ax.set_xticklabels([f'{n}×' for n in range(7)])

axes[-1].set_xlabel('Order (× running speed)', fontsize=11)
fig.suptitle('Fault Diagnostic Fingerprints — FFT Order Spectra\n'
             f'Jeffcott Rotor | {CONFIG["rpm"]:.0f} rpm | ζ = {CONFIG["zeta"]:.2f}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_fault_fingerprints.png', bbox_inches='tight')
plt.show()

print('\n' + '='*60)
print('  FAULT DIAGNOSTIC SUMMARY')
print('='*60)
print('  Fault Type        | Key Spectral Feature      | Orbit Shape')
print('  ------------------|---------------------------|------------')
print('  Healthy           | 1× only                   | Circle')
print('  Mass Unbalance    | Elevated 1×               | Large circle')
print('  Breathing Crack   | 1× + visible 2×           | Inner loop')
print('  Bearing Clearance | 1× + 2× + 3× + harmonics | Clipped/flat')
print('='*60)
print()
print('Simulation complete')